# Ultimate Incurred Claim Cost Prediction
## Workers' Compensation Insurance — Actuarial NLP + Machine Learning Pipeline

---

### Project Overview

This notebook implements a **reproducible, end-to-end actuarial machine learning pipeline** to predict `UltimateIncurredClaimCost` from workers' compensation claims data. It combines **structured claim variables** with **unstructured text embeddings** derived from `ClaimDescription` fields.

The methodology follows modern insurance analytics best practices including:
- Strict **data leakage prevention** (no target-derived features in training)
- **BERT-based sentence embeddings** for NLP feature extraction
- **Unsupervised clustering** to create semantically meaningful claim categories
- **XGBoost** gradient boosting with cross-validated hyperparameter tuning
- **SHAP** explainability for both global and individual claim interpretation

---

### Pipeline Architecture

```
Raw Data (CSV)
     │
     ├─── Structured Features ──── Preprocessing & Engineering ────────────┐
     │                                                                       │
     └─── ClaimDescription ──── BERT Embeddings ──── UMAP ──── K-Means ───┘
                                                                             │
                                                              XGBoost Model  │
                                                              (CV-Tuned)  ◄──┘
                                                                    │
                                                    ┌───────────────┴──────────────┐
                                               Evaluation                      SHAP XAI
                                           (RMSE/MAE/R²)             (Global + Per-Claim)
```

---

### Dataset

| Field | Description |
|---|---|
| `ClaimNumber` | Unique claim identifier (excluded from modeling) |
| `DateTimeOfAccident` | Timestamp of the workplace accident |
| `DateReported` | Date the claim was filed |
| `Age` | Claimant age at time of accident |
| `Gender` | Claimant gender |
| `MaritalStatus` | Claimant marital status |
| `DependentChildren` / `DependentsOther` | Number of dependents |
| `WeeklyWages` | Claimant pre-injury weekly wages |
| `PartTimeFullTime` | Employment type |
| `HoursWorkedPerWeek` / `DaysWorkedPerWeek` | Work schedule |
| `ClaimDescription` | Free-text description of the injury |
| `InitialIncurredCalimsCost` | ⚠️ Early cost estimate — used cautiously (see leakage section) |
| `UltimateIncurredClaimCost` | **Target variable** — final settled claim cost |

---

### Reproducibility

All random seeds are set to `42`. Package versions are pinned where possible.  
**Runtime:** Set Colab to `GPU` (Runtime → Change runtime type → T4 GPU) for faster BERT inference.

**Author:** Actuarial Data Science Research  
**Date:** 2024  
**R Version:** 4.3+

---
## Section 0 — Environment Setup

We install and load all required R packages and configure the Python environment via `reticulate`.  
Packages are only installed if not already present to minimise repeated installation time.

> **Note on reticulate:** The `reticulate` package creates a bridge between R and Python, allowing us to call `sentence-transformers` (a Python library) directly from R. BERT embeddings are generated in Python and returned to R as a numeric matrix.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0.1 — Install R Packages
# We use a helper function to install only missing packages, avoiding
# re-installation on every session restart.
# ─────────────────────────────────────────────────────────────────────────────

install_if_missing <- function(pkgs) {
  missing <- pkgs[!pkgs %in% installed.packages()[, "Package"]]
  if (length(missing) > 0) {
    message("Installing: ", paste(missing, collapse = ", "))
    install.packages(missing, repos = "https://cran.rstudio.com/", quiet = TRUE)
  } else {
    message("All packages already installed.")
  }
}

r_packages <- c(
  # ── Core data manipulation ──────────────────────────────────────────────────
  "tidyverse",    # dplyr, ggplot2, tidyr, stringr, forcats, purrr
  "data.table",   # fast data reading and manipulation
  "lubridate",    # date/time parsing and arithmetic

  # ── Text processing ─────────────────────────────────────────────────────────
  "text2vec",     # TF-IDF and GloVe baselines (for comparison)
  "reticulate",   # Python interop — required for BERT via sentence-transformers

  # ── Clustering ──────────────────────────────────────────────────────────────
  "cluster",      # silhouette scores and PAM clustering
  "factoextra",   # cluster visualisation and elbow plots
  "umap",         # UMAP dimensionality reduction

  # ── Modelling ───────────────────────────────────────────────────────────────
  "xgboost",      # gradient boosted trees
  "Matrix",       # sparse matrix support for xgboost DMatrix
  "caret",        # cross-validation and preprocessing utilities

  # ── Explainability ──────────────────────────────────────────────────────────
  "shapviz",          # SHAP waterfall, beeswarm, dependence plots
  "SHAPforxgboost",   # SHAP computation and long-format prep for ggplot

  # ── Visualisation helpers ───────────────────────────────────────────────────
  "scales",       # axis formatting (dollar, comma, percent)
  "gridExtra",    # multi-panel plot layouts
  "corrplot",     # correlation matrix heatmaps
  "viridis",      # perceptually uniform colour scales
  "knitr",        # formatted table output
  "kableExtra"    # enhanced HTML/LaTeX tables
)

install_if_missing(r_packages)

# Load all packages, suppressing startup messages for cleaner output
suppressPackageStartupMessages(
  invisible(lapply(r_packages, library, character.only = TRUE))
)

cat("\n✅ All R packages loaded successfully.\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0.2 — Python Environment Configuration
#
# We use reticulate to access Python's sentence-transformers library for BERT.
# The system Python in Google Colab already has torch and numpy; we add the
# remaining NLP packages via pip.
# ─────────────────────────────────────────────────────────────────────────────

library(reticulate)

# Point reticulate to Colab's system Python (which has GPU support)
use_python("/usr/bin/python3", required = TRUE)

# Install Python NLP and clustering packages
# -q = quiet mode; installs only if not already present
py_packages <- c(
  "sentence-transformers",  # BERT / transformer sentence embeddings
  "hdbscan",                # density-based clustering (alternative to K-Means)
  "umap-learn",             # UMAP in Python (for embedding reduction)
  "scikit-learn"            # silhouette scoring, PCA
)

system(paste("pip install -q", paste(py_packages, collapse = " ")))

cat("✅ Python packages installed.\n")
cat("Python version: ", py_run_string("import sys; print(sys.version)")$output, "\n")

# Verify GPU availability
py_run_string("
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0.3 — Global Configuration
#
# Centralise all configuration: seed, paths, and key model parameters.
# This makes the notebook easy to adapt for different datasets or experiments.
# ─────────────────────────────────────────────────────────────────────────────

# ── Reproducibility seed (applied to R and passed to Python) ─────────────────
GLOBAL_SEED <- 42L
set.seed(GLOBAL_SEED)

# ── File paths ────────────────────────────────────────────────────────────────
# Change DATA_PATH if your file is in Google Drive or uploaded elsewhere
CONFIG <- list(
  data_path     = "data.csv",          # input data
  model_path    = "xgb_model.json",    # saved XGBoost model
  pred_path     = "predictions.csv",   # saved predictions
  train_frac    = 0.80,                # 80/20 train-test split
  cv_folds      = 5L,                  # k-fold cross-validation
  early_stop    = 30L,                 # early stopping rounds
  max_nrounds   = 600L,                # max boosting iterations
  shap_n_sample = 5000L,               # SHAP sample size (memory/speed trade-off)
  bert_model    = "all-MiniLM-L6-v2",  # fast 384-dim BERT model
  bert_batch    = 256L,                # BERT batch size
  umap_2d_neighbors  = 15L,           # UMAP 2D: for visualisation
  umap_10d_neighbors = 15L,           # UMAP 10D: for clustering
  kmeans_k_max       = 12L            # max K to evaluate in elbow search
)

# ── ggplot2 theme for all plots ───────────────────────────────────────────────
theme_actuarial <- function(base_size = 12) {
  theme_minimal(base_size = base_size) +
  theme(
    plot.title       = element_text(face = "bold", size = base_size + 2),
    plot.subtitle    = element_text(colour = "grey40", size = base_size - 1),
    plot.caption     = element_text(colour = "grey60", size = base_size - 3),
    panel.grid.minor = element_blank(),
    legend.position  = "bottom",
    strip.text       = element_text(face = "bold")
  )
}

# Set as default
theme_set(theme_actuarial())

cat("✅ Global configuration set. Seed:", GLOBAL_SEED, "\n")
cat("   Data path:", CONFIG$data_path, "\n")
cat("   BERT model:", CONFIG$bert_model, "\n")

---
## Section 1 — Data Loading & Initial Inspection

We load the data and perform a structured **exploratory data analysis (EDA)** before any transformations. This step:
- Establishes the raw data shape and types
- Identifies missing values and their distribution
- Examines the target variable distribution (skewness is common in claim costs)
- Checks for data quality issues before any preprocessing

> **Actuarial note:** Ultimate claim cost distributions are typically **heavy-tailed and right-skewed**. This is a key modelling consideration — we will apply a log transformation to stabilise variance and improve model fit.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1.1 — Load Data
#
# We use data.table::fread() for fast reading, then convert to a tibble
# for tidyverse compatibility. fread() auto-detects types and handles
# large files more efficiently than read.csv().
# ─────────────────────────────────────────────────────────────────────────────

# ── Option A: Direct file path (default) ─────────────────────────────────────
df_raw <- data.table::fread(CONFIG$data_path) |> as_tibble()

# ── Option B: Upload interactively in Colab ───────────────────────────────────
# Uncomment to use file upload widget:
# py_run_string("from google.colab import files; uploaded = files.upload()")
# df_raw <- data.table::fread(names(py$uploaded)[1]) |> as_tibble()

# ── Option C: Load from Google Drive ─────────────────────────────────────────
# py_run_string("from google.colab import drive; drive.mount('/content/drive')")
# df_raw <- data.table::fread("/content/drive/MyDrive/data.csv") |> as_tibble()

cat("══════════════════════════════════════════\n")
cat("  DATASET LOADED\n")
cat("══════════════════════════════════════════\n")
cat("  Rows:   ", scales::comma(nrow(df_raw)), "\n")
cat("  Columns:", ncol(df_raw), "\n")
cat("══════════════════════════════════════════\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1.2 — Data Structure & Types
#
# glimpse() gives a transposed summary showing each column's type and
# sample values. This is the first sanity check before any processing.
# ─────────────────────────────────────────────────────────────────────────────

cat("Column types and sample values:\n\n")
glimpse(df_raw)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1.3 — Summary Statistics
#
# We compute a formatted summary table for numeric columns including quartiles,
# mean, min, max, and NA count to understand distributions before modelling.
# ─────────────────────────────────────────────────────────────────────────────

cat("── Summary Statistics (Numeric Columns) ──\n")
df_raw |>
  select(where(is.numeric)) |>
  summary() |>
  print()

cat("\n── Categorical Column Distributions ──\n")
cat("\nGender:\n");        print(table(df_raw$Gender,        useNA = "ifany"))
cat("\nMaritalStatus:\n"); print(table(df_raw$MaritalStatus, useNA = "ifany"))
cat("\nPartTimeFullTime:\n"); print(table(df_raw$PartTimeFullTime, useNA = "ifany"))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1.4 — Missing Value Analysis
#
# Missing values are computed per column as both counts and percentages.
# A bar chart visualises the severity of missingness. We need this before
# deciding on imputation strategy.
# ─────────────────────────────────────────────────────────────────────────────

missing_tbl <- df_raw |>
  summarise(across(everything(), ~ sum(is.na(.)))) |>
  pivot_longer(everything(), names_to = "column", values_to = "n_missing") |>
  mutate(
    pct_missing = round(n_missing / nrow(df_raw) * 100, 2),
    type        = if_else(pct_missing > 0, "Has Missing", "Complete")
  ) |>
  arrange(desc(n_missing))

cat("── Missing Value Summary ──\n")
print(missing_tbl)

# Visualise — only show columns with missing data
missing_plot_data <- missing_tbl |> filter(n_missing > 0)

if (nrow(missing_plot_data) > 0) {
  ggplot(missing_plot_data,
         aes(x = reorder(column, -pct_missing), y = pct_missing, fill = pct_missing)) +
    geom_col(width = 0.7) +
    geom_text(aes(label = paste0(pct_missing, "%")),
              vjust = -0.4, size = 3.5, colour = "grey30") +
    scale_fill_gradient(low = "#FFF176", high = "#C62828", guide = "none") +
    scale_y_continuous(labels = scales::percent_format(scale = 1)) +
    labs(
      title    = "Missing Value Analysis",
      subtitle = "Columns with at least one missing observation",
      x        = NULL,
      y        = "% Missing",
      caption  = paste("Total rows:", scales::comma(nrow(df_raw)))
    ) +
    theme(axis.text.x = element_text(angle = 40, hjust = 1))
} else {
  cat("\n✅ No missing values detected in the dataset.\n")
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1.5 — Target Variable Distribution
#
# UltimateIncurredClaimCost is the response variable. Insurance claim costs
# are almost universally right-skewed and heavy-tailed. We visualise both
# the raw and log-transformed distributions to confirm this and justify
# modelling on the log scale.
# ─────────────────────────────────────────────────────────────────────────────

target_stats <- df_raw |>
  summarise(
    n       = n(),
    mean    = mean(UltimateIncurredClaimCost, na.rm = TRUE),
    median  = median(UltimateIncurredClaimCost, na.rm = TRUE),
    sd      = sd(UltimateIncurredClaimCost, na.rm = TRUE),
    p25     = quantile(UltimateIncurredClaimCost, 0.25, na.rm = TRUE),
    p75     = quantile(UltimateIncurredClaimCost, 0.75, na.rm = TRUE),
    p95     = quantile(UltimateIncurredClaimCost, 0.95, na.rm = TRUE),
    p99     = quantile(UltimateIncurredClaimCost, 0.99, na.rm = TRUE),
    max_val = max(UltimateIncurredClaimCost, na.rm = TRUE),
    skew    = (mean - median) / sd  # Pearson's moment coefficient
  )

cat("── Target Variable Statistics ──\n")
cat(sprintf("  Mean:         $%s\n", scales::comma(round(target_stats$mean, 0))))
cat(sprintf("  Median:       $%s\n", scales::comma(round(target_stats$median, 0))))
cat(sprintf("  Std Dev:      $%s\n", scales::comma(round(target_stats$sd, 0))))
cat(sprintf("  P95:          $%s\n", scales::comma(round(target_stats$p95, 0))))
cat(sprintf("  P99:          $%s\n", scales::comma(round(target_stats$p99, 0))))
cat(sprintf("  Maximum:      $%s\n", scales::comma(round(target_stats$max_val, 0))))
cat(sprintf("  Skewness:     %.4f\n", target_stats$skew))

# ── Plot: raw vs log-transformed ──────────────────────────────────────────────
p_raw <- ggplot(df_raw, aes(x = UltimateIncurredClaimCost)) +
  geom_histogram(bins = 100, fill = "#1565C0", alpha = 0.85, colour = "white") +
  geom_vline(xintercept = target_stats$mean,   colour = "#E53935", linetype = "dashed", linewidth = 0.8) +
  geom_vline(xintercept = target_stats$median, colour = "#2E7D32", linetype = "dotted", linewidth = 0.8) +
  annotate("text", x = target_stats$mean * 1.1,   y = Inf, label = "Mean",   vjust = 2, colour = "#E53935", size = 3) +
  annotate("text", x = target_stats$median * 1.1, y = Inf, label = "Median", vjust = 3.5, colour = "#2E7D32", size = 3) +
  scale_x_continuous(labels = scales::dollar_format(scale = 0.001, suffix = "K")) +
  labs(
    title    = "Raw Target Distribution",
    subtitle = "Heavy right tail typical of insurance claims",
    x        = "Ultimate Claim Cost",
    y        = "Count"
  )

p_log <- ggplot(df_raw, aes(x = log1p(UltimateIncurredClaimCost))) +
  geom_histogram(bins = 80, fill = "#2E7D32", alpha = 0.85, colour = "white") +
  geom_density(aes(y = after_stat(count)), colour = "#E53935", linewidth = 0.8) +
  labs(
    title    = "Log-Transformed Target Distribution",
    subtitle = "log1p(x) stabilises variance — more symmetric",
    x        = "log1p(UltimateIncurredClaimCost)",
    y        = "Count"
  )

gridExtra::grid.arrange(p_raw, p_log, nrow = 1,
  top = gridExtra::textGrob(
    "Target Variable: UltimateIncurredClaimCost",
    gp = grid::gpar(fontface = "bold", fontsize = 14)
  )
)

---
## Section 2 — Data Leakage Prevention

**Data leakage** occurs when information from the test set or future information contaminates model training, leading to unrealistically optimistic evaluation metrics.

### Leakage Assessment for This Dataset

| Feature | Leakage Risk | Decision |
|---|---|---|
| `ClaimNumber` | None (identifier) | ❌ Excluded entirely |
| `UltimateIncurredClaimCost` | **Target** | ✅ Only as label |
| `InitialIncurredCalimsCost` | ⚠️ **High Risk** | ✅ Used — but with caution (see below) |
| `DateReported`, `DateTimeOfAccident` | Low (known at report time) | ✅ Temporal features derived safely |
| All other features | None | ✅ Safe to use |

### Regarding `InitialIncurredCalimsCost`

This column is an **early cost reserve** set by the insurer at claim registration — it is **not** a derived or future feature, but represents information available at underwriting time. However, it is highly correlated with the target by construction. We include it but:
1. Log-transform it to reduce its dominance
2. Monitor it in SHAP analysis to ensure it isn't overshadowing genuinely predictive features
3. If deploying for pricing (where initial cost is unknown), this feature would be excluded

> **Key rule:** All preprocessing steps (imputation means, scaling parameters) are computed **only on the training set** and then applied to the test set. This is enforced explicitly in Section 5.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2.1 — Leakage Audit: Correlation with Target
#
# Before preprocessing, we compute correlations between all numeric features
# and the target. Features with near-perfect correlation may signal leakage.
# ─────────────────────────────────────────────────────────────────────────────

leakage_check <- df_raw |>
  select(where(is.numeric)) |>
  select(-UltimateIncurredClaimCost) |>   # exclude target
  cor(df_raw$UltimateIncurredClaimCost, use = "complete.obs") |>
  as.data.frame() |>
  rownames_to_column("feature") |>
  rename(corr_with_target = V1) |>
  arrange(desc(abs(corr_with_target)))

cat("── Pearson Correlation with UltimateIncurredClaimCost ──\n")
print(leakage_check)

# Threshold warning: correlation > 0.95 may indicate leakage
high_corr <- leakage_check |> filter(abs(corr_with_target) > 0.95)
if (nrow(high_corr) > 0) {
  cat("\n⚠️  WARNING: Potential leakage features (|r| > 0.95):\n")
  print(high_corr)
} else {
  cat("\n✅ No extreme correlations detected (|r| > 0.95). Leakage risk is low.\n")
}

---
## Section 3 — Feature Engineering & Preprocessing

We engineer informative features from the raw columns before modelling.

### Feature Engineering Plan

| New Feature | Source | Rationale |
|---|---|---|
| `ReportingDelay` | `DateReported − DateTimeOfAccident` | IBNR proxy; long delays ↔ complex claims |
| `AccidentYear` | `DateTimeOfAccident` | Trend / development year |
| `AccidentMonth` | `DateTimeOfAccident` | Seasonal patterns |
| `AccidentWeekday` | `DateTimeOfAccident` | Mon–Fri claims differ from weekend |
| `AccidentHour` | `DateTimeOfAccident` | Shift-related injury patterns |
| `IsWeekend` | `AccidentWeekday` | Binary flag — weekend accidents differ |
| `log_WeeklyWages` | `WeeklyWages` | Log-transform — wage distributions are right-skewed |
| `log_InitialCost` | `InitialIncurredCalimsCost` | Log-transform — see leakage section |

> **Preprocessing order matters:** Imputation statistics are computed only on training data to prevent leakage. We implement this via a two-step approach: first define the preprocessing on the full dataset for feature structure, then re-fit on train only before applying to test.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3.1 — Date Parsing & Temporal Feature Engineering
#
# lubridate parses ISO 8601 timestamps reliably across time zones.
# All temporal features are derived from the accident/report timestamps
# that would be available at claim intake — no future information used.
# ─────────────────────────────────────────────────────────────────────────────

df <- df_raw |>
  mutate(
    # Parse timestamps (ISO 8601 format with UTC timezone)
    dt_accident  = lubridate::ymd_hms(DateTimeOfAccident, quiet = TRUE),
    dt_reported  = lubridate::ymd_hms(DateReported,       quiet = TRUE),

    # ── Reporting delay (days) ──────────────────────────────────────────────
    # Actuarial significance: longer delays often indicate more severe or
    # contested claims (tail claims), or late-reported occupational diseases.
    ReportingDelay = as.numeric(
      difftime(dt_reported, dt_accident, units = "days")
    ),

    # ── Accident year (development period proxy) ────────────────────────────
    AccidentYear    = lubridate::year(dt_accident),

    # ── Accident month (seasonal effects: construction, agriculture) ────────
    AccidentMonth   = lubridate::month(dt_accident),

    # ── Weekday (1=Sun … 7=Sat) — Mon injuries may be Monday Effect claims ─
    AccidentWeekday = lubridate::wday(dt_accident, label = FALSE),

    # ── Hour of accident (shift patterns: early morning, overnight) ─────────
    AccidentHour    = lubridate::hour(dt_accident),

    # ── Weekend flag (binary) ───────────────────────────────────────────────
    IsWeekend       = as.integer(AccidentWeekday %in% c(1L, 7L))
  )

cat("── Temporal Features Summary ──\n")
cat(sprintf("  Accident years:  %d – %d\n",
            min(df$AccidentYear, na.rm = TRUE),
            max(df$AccidentYear, na.rm = TRUE)))
cat(sprintf("  Reporting delay: %.0f to %.0f days (median: %.0f)\n",
            min(df$ReportingDelay, na.rm = TRUE),
            max(df$ReportingDelay, na.rm = TRUE),
            median(df$ReportingDelay, na.rm = TRUE)))

# Distribution of reporting delay
ggplot(df |> filter(ReportingDelay >= 0, ReportingDelay <= 365),
       aes(x = ReportingDelay)) +
  geom_histogram(bins = 60, fill = "#5C6BC0", alpha = 0.85, colour = "white") +
  geom_vline(xintercept = median(df$ReportingDelay, na.rm = TRUE),
             colour = "#E53935", linetype = "dashed", linewidth = 0.9) +
  scale_x_continuous(breaks = seq(0, 365, by = 30)) +
  labs(
    title    = "Reporting Delay Distribution",
    subtitle = "Days between accident and claim filing (capped at 365 for display)",
    x        = "Reporting Delay (days)",
    y        = "Count",
    caption  = "Red dashed line = median"
  )

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3.2 — Missing Value Imputation
#
# Strategy:
#   • Numeric columns → median imputation (robust to outliers)
#   • Categorical columns → mode imputation or 'Unknown' category
#   • ClaimDescription → empty string (handled downstream by BERT)
#
# LEAKAGE NOTE: Imputation statistics (medians) are recorded here for
# documentation purposes. In Section 5, they are recomputed from
# TRAINING data only before applying to the test set.
# ─────────────────────────────────────────────────────────────────────────────

# Compute imputation values from the full dataset (updated to train-only in §5)
impute_vals <- list(
  Age               = median(df$Age,               na.rm = TRUE),
  WeeklyWages       = median(df$WeeklyWages,        na.rm = TRUE),
  HoursWorkedPerWeek= median(df$HoursWorkedPerWeek, na.rm = TRUE),
  DaysWorkedPerWeek = median(df$DaysWorkedPerWeek,  na.rm = TRUE),
  ReportingDelay    = median(df$ReportingDelay,     na.rm = TRUE),
  InitialCost       = median(df$InitialIncurredCalimsCost, na.rm = TRUE)
)

df <- df |>
  mutate(
    # Numeric imputation with medians
    Age                = coalesce(Age,                impute_vals$Age),
    WeeklyWages        = coalesce(WeeklyWages,        impute_vals$WeeklyWages),
    HoursWorkedPerWeek = coalesce(HoursWorkedPerWeek, impute_vals$HoursWorkedPerWeek),
    DaysWorkedPerWeek  = coalesce(DaysWorkedPerWeek,  impute_vals$DaysWorkedPerWeek),
    ReportingDelay     = coalesce(ReportingDelay,     impute_vals$ReportingDelay),
    InitialIncurredCalimsCost = coalesce(
      InitialIncurredCalimsCost, impute_vals$InitialCost
    ),

    # Categorical imputation
    Gender           = coalesce(Gender,           "Unknown"),
    MaritalStatus    = coalesce(MaritalStatus,    "Unknown"),
    PartTimeFullTime = coalesce(PartTimeFullTime, "Unknown"),

    # Text: empty string allows BERT to produce a low-magnitude embedding
    ClaimDescription = coalesce(ClaimDescription, "")
  )

# Verify no remaining NAs in key columns
remaining_na <- df |>
  select(-ClaimNumber, -DateTimeOfAccident, -DateReported,
         -dt_accident, -dt_reported) |>
  summarise(across(everything(), ~ sum(is.na(.)))) |>
  pivot_longer(everything()) |>
  filter(value > 0)

if (nrow(remaining_na) == 0) {
  cat("✅ Imputation complete. No remaining missing values in model features.\n")
} else {
  cat("⚠️  Remaining NAs after imputation:\n")
  print(remaining_na)
}

cat("\n── Imputed Medians (from full dataset — refit on train in §5) ──\n")
purrr::iwalk(impute_vals, ~ cat(sprintf("  %-25s %s\n", .y, scales::comma(round(.x, 2)))))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3.3 — Categorical Encoding
#
# XGBoost requires numeric input. We integer-encode categorical variables.
# Encoding maps: stored so they can be applied consistently to new data.
#
# Note: We do NOT use one-hot encoding here because XGBoost natively handles
# ordinal/integer categories via tree splits without creating sparse matrices.
# ─────────────────────────────────────────────────────────────────────────────

# Define encoding maps (alphabetically consistent)
gender_levels       <- sort(unique(df$Gender))
marital_levels      <- sort(unique(df$MaritalStatus))
parttime_levels     <- sort(unique(df$PartTimeFullTime))

df <- df |>
  mutate(
    Gender_enc           = as.integer(factor(Gender,           levels = gender_levels)),
    MaritalStatus_enc    = as.integer(factor(MaritalStatus,    levels = marital_levels)),
    PartTimeFullTime_enc = as.integer(factor(PartTimeFullTime, levels = parttime_levels))
  )

# Log-transform skewed continuous features
# log1p(x) = log(1+x) handles x=0 gracefully and is easily inverted with expm1()
df <- df |>
  mutate(
    log_WeeklyWages  = log1p(WeeklyWages),
    log_InitialCost  = log1p(InitialIncurredCalimsCost),
    log_Target       = log1p(UltimateIncurredClaimCost)  # model target
  )

# Record encoding maps for reproducibility / future scoring
encoding_maps <- list(
  Gender           = setNames(seq_along(gender_levels),   gender_levels),
  MaritalStatus    = setNames(seq_along(marital_levels),  marital_levels),
  PartTimeFullTime = setNames(seq_along(parttime_levels), parttime_levels)
)

cat("── Categorical Encoding Maps ──\n")
purrr::iwalk(encoding_maps, function(map, nm) {
  cat(sprintf("\n%s:\n", nm))
  cat(paste0("  ", names(map), " → ", map, collapse = "\n"), "\n")
})

cat("\n✅ Encoding complete. Sample encoded values:\n")
df |>
  select(Gender, Gender_enc, MaritalStatus, MaritalStatus_enc,
         log_WeeklyWages, log_InitialCost, log_Target) |>
  head(5) |>
  print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3.4 — Exploratory Feature Visualisations
# ─────────────────────────────────────────────────────────────────────────────

# Numeric correlation heatmap
num_features <- df |>
  select(Age, log_WeeklyWages, HoursWorkedPerWeek, DaysWorkedPerWeek,
         DependentChildren, DependentsOther, ReportingDelay,
         AccidentMonth, AccidentWeekday, AccidentHour, AccidentYear,
         log_InitialCost, log_Target)

cor_matrix <- cor(num_features, use = "complete.obs")

corrplot::corrplot(
  cor_matrix,
  method     = "color",
  type       = "upper",
  tl.cex     = 0.80,
  tl.col     = "black",
  col        = corrplot::COL2("RdBu", 200),
  addCoef.col= "black",
  number.cex = 0.55,
  title      = "Correlation Matrix — Numeric Features",
  mar        = c(0, 0, 2, 0)
)

In [ ]:
# ── Feature distributions by categorical groups ───────────────────────────────
p_gender <- df |>
  filter(Gender %in% c("M", "F")) |>
  ggplot(aes(x = Gender, y = log_Target, fill = Gender)) +
  geom_boxplot(alpha = 0.7, outlier.size = 0.5) +
  scale_fill_manual(values = c("F" = "#E91E63", "M" = "#1565C0")) +
  labs(title = "Claim Cost by Gender", x = NULL, y = "log1p(Claim Cost)") +
  theme(legend.position = "none")

p_ptype <- ggplot(df |> filter(PartTimeFullTime %in% c("F", "P")),
                  aes(x = PartTimeFullTime, y = log_Target, fill = PartTimeFullTime)) +
  geom_violin(alpha = 0.6) +
  geom_boxplot(width = 0.15, fill = "white", outlier.size = 0.3) +
  scale_fill_manual(values = c("F" = "#2E7D32", "P" = "#F57F17")) +
  labs(title = "Claim Cost: Full vs Part Time", x = NULL, y = "log1p(Claim Cost)") +
  theme(legend.position = "none")

p_month <- df |>
  group_by(AccidentMonth) |>
  summarise(median_cost = median(UltimateIncurredClaimCost), .groups = "drop") |>
  ggplot(aes(x = factor(AccidentMonth), y = median_cost)) +
  geom_col(fill = "#5C6BC0", alpha = 0.85) +
  scale_y_continuous(labels = scales::dollar_format()) +
  scale_x_discrete(labels = month.abb) +
  labs(title = "Median Claim Cost by Accident Month",
       x = "Month", y = "Median Cost")

p_delay <- ggplot(df |> filter(ReportingDelay >= 0, ReportingDelay <= 180),
                  aes(x = ReportingDelay, y = log_Target)) +
  geom_hex(bins = 50) +
  scale_fill_viridis_c(option = "plasma") +
  labs(title = "Claim Cost vs Reporting Delay",
       x = "Reporting Delay (days)", y = "log1p(Claim Cost)",
       fill = "Count")

gridExtra::grid.arrange(p_gender, p_ptype, p_month, p_delay, nrow = 2)

---
## Section 4 — BERT Sentence Embeddings

We use a **pre-trained transformer model** (`all-MiniLM-L6-v2`) to convert each `ClaimDescription` into a dense 384-dimensional vector.

### Why BERT over TF-IDF?

| Method | Captures Semantics | Handles Spelling | Dimension | Notes |
|---|---|---|---|---|
| TF-IDF (text2vec) | ❌ Bag-of-words only | ❌ | High (vocab size) | Fast, interpretable |
| Word2Vec/GloVe | ⚠️ Word-level only | ❌ | 100–300 | No context |
| **BERT (MiniLM)** | ✅ Full sentence context | ✅ Subword tokens | 384 | Best semantic quality |

### Model: `all-MiniLM-L6-v2`
- **384 dimensions** — compact but semantically rich
- Trained on 1 billion sentence pairs via contrastive learning
- Optimised for **semantic similarity** tasks (ideal for injury description clustering)
- Runs at ~14,000 sentences/second on GPU

Embeddings are normalised to unit length (cosine-distance compatible).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4.1 — Prepare Texts for Embedding
#
# Light cleaning only — BERT tokenisation handles most normalisation internally.
# We avoid aggressive stemming/stopword removal as BERT uses full context.
# ─────────────────────────────────────────────────────────────────────────────

# Clean text: uppercase, collapse whitespace, remove non-ASCII artefacts
claim_texts <- df$ClaimDescription |>
  stringr::str_to_upper() |>       # standardise case
  stringr::str_squish() |>         # remove leading/trailing/extra whitespace
  stringr::str_replace_all("[^[:print:]]", " ") |>  # remove non-printable
  replace_na("")                   # empty string for missing descriptions

# Export to Python namespace
py$claim_texts  <- claim_texts
py$bert_model   <- CONFIG$bert_model
py$bert_batch   <- CONFIG$bert_batch
py$global_seed  <- GLOBAL_SEED

cat("── Text Sample After Cleaning ──\n")
cat(sprintf("  Total texts: %s\n", scales::comma(length(claim_texts))))
cat(sprintf("  Empty texts: %d\n", sum(claim_texts == "")))
cat("\nSample descriptions:\n")
head(claim_texts, 6) |> purrr::walk(~ cat(sprintf("  · %s\n", .x)))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4.2 — Generate BERT Embeddings (Python via reticulate)
#
# sentence-transformers handles batching, tokenisation, and pooling internally.
# normalize_embeddings=True → L2 unit norm → cosine similarity = dot product
# ─────────────────────────────────────────────────────────────────────────────

py_run_string("
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# ── Device selection ────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Computing embeddings on: {device.upper()}')

# ── Load pre-trained model ──────────────────────────────────────────────────
# all-MiniLM-L6-v2: 6-layer distilled BERT, 384-dim, 22M parameters
# Excellent semantic quality with minimal memory footprint
print(f'Loading model: {bert_model}')
encoder = SentenceTransformer(bert_model, device=device)

# ── Generate embeddings ─────────────────────────────────────────────────────
print(f'Encoding {len(claim_texts):,} claim descriptions in batches of {bert_batch}...')
embeddings = encoder.encode(
    claim_texts,
    batch_size          = bert_batch,
    show_progress_bar   = True,
    normalize_embeddings= True,   # L2-norm → unit sphere
    device              = device
)

print(f'Embedding matrix: {embeddings.shape[0]:,} rows × {embeddings.shape[1]} dims')
print(f'Memory: {embeddings.nbytes / 1e6:.1f} MB')
print(f'Embedding norm check (should be ~1.0): {np.linalg.norm(embeddings[0]):.6f}')
")

# ── Pull embeddings back to R ─────────────────────────────────────────────────
bert_embeddings <- py$embeddings
cat(sprintf("\n✅ Embeddings in R: %d × %d matrix\n",
            nrow(bert_embeddings), ncol(bert_embeddings)))

---
## Section 5 — Unsupervised Text Clustering

We cluster the BERT embeddings to create a categorical `TextCluster` feature that captures **semantic injury type patterns** from the free-text descriptions.

### Why Cluster?
Rather than feeding raw 384-dimensional embeddings directly into XGBoost (curse of dimensionality), we:
1. Reduce to 10D via UMAP (preserves global and local structure)
2. Apply K-Means to obtain discrete clusters (actuarially interpretable categories)
3. Add the cluster label as a single categorical feature

This approach creates **semantically coherent injury groups** (e.g., back strains, fractures, lacerations) that have different cost profiles.

### Clustering Pipeline
```
BERT (384D) → UMAP (10D) → K-Means → TextCluster label
                 ↓
             UMAP (2D) → Visualisation
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5.1 — UMAP Dimensionality Reduction
#
# Two UMAP projections:
#   • 2D  → human-interpretable scatter plots
#   • 10D → clustering (more informative than 2D, less noisy than 384D)
#
# Parameters:
#   • n_neighbors: balance between local and global structure
#   • min_dist=0.0 (10D): allows tighter clusters
#   • min_dist=0.1 (2D): prevents over-crowding in plots
#   • metric='cosine': appropriate for L2-normalised BERT vectors
# ─────────────────────────────────────────────────────────────────────────────

py$embeddings_np    <- bert_embeddings
py$umap_2d_nn       <- CONFIG$umap_2d_neighbors
py$umap_10d_nn      <- CONFIG$umap_10d_neighbors

py_run_string("
import umap
import numpy as np

# ── 2D UMAP for visualisation ───────────────────────────────────────────────
print('UMAP 2D (visualisation)...')
reducer_2d = umap.UMAP(
    n_components = 2,
    n_neighbors  = umap_2d_nn,
    min_dist     = 0.1,
    metric       = 'cosine',
    random_state = global_seed,
    low_memory   = True
)
umap_2d = reducer_2d.fit_transform(embeddings_np)
print(f'  Done. Shape: {umap_2d.shape}')

# ── 10D UMAP for clustering ─────────────────────────────────────────────────
# min_dist=0.0 forces tighter clusters in the embedding space
print('UMAP 10D (clustering)...')
reducer_10d = umap.UMAP(
    n_components = 10,
    n_neighbors  = umap_10d_nn,
    min_dist     = 0.0,
    metric       = 'cosine',
    random_state = global_seed,
    low_memory   = True
)
umap_10d = reducer_10d.fit_transform(embeddings_np)
print(f'  Done. Shape: {umap_10d.shape}')
")

umap_2d  <- py$umap_2d
umap_10d <- py$umap_10d

cat("✅ UMAP projections complete.\n")
cat(sprintf("  2D shape:  %d × %d\n", nrow(umap_2d),  ncol(umap_2d)))
cat(sprintf("  10D shape: %d × %d\n", nrow(umap_10d), ncol(umap_10d)))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5.2 — Optimal K Selection (Elbow + Silhouette)
#
# We evaluate K=2..12 using:
#   1. Total Within-Cluster Sum of Squares (WCSS) — elbow method
#   2. Average Silhouette Score — measures cluster separation quality
#      Score ranges from -1 (poor) to +1 (excellent). Values > 0.5 are good.
#
# A random subsample of 5,000 observations is used for silhouette computation
# (silhouette is O(n²) — full dataset would be too slow).
# ─────────────────────────────────────────────────────────────────────────────

set.seed(GLOBAL_SEED)
K_MAX     <- CONFIG$kmeans_k_max
samp_n    <- 5000L
samp_idx  <- sample(nrow(umap_10d), min(samp_n, nrow(umap_10d)))
umap_samp <- umap_10d[samp_idx, ]

# Storage for metrics
cluster_metrics <- tibble(
  k         = 2:K_MAX,
  wcss      = NA_real_,
  silhouette = NA_real_
)

cat(sprintf("Evaluating K = 2 to %d on %s-observation sample...\n",
            K_MAX, scales::comma(samp_n)))
cat(sprintf("%-6s %-15s %-15s\n", "K", "WCSS", "Silhouette"))
cat(strrep("-", 40), "\n")

for (k in 2:K_MAX) {
  km  <- kmeans(umap_samp, centers = k, nstart = 15L, iter.max = 150L)
  sil <- cluster::silhouette(km$cluster, dist(umap_samp))

  cluster_metrics$wcss[k - 1]       <- km$tot.withinss
  cluster_metrics$silhouette[k - 1] <- mean(sil[, 3])

  cat(sprintf("K=%-4d  WCSS=%-12.1f  Sil=%.4f\n",
              k, km$tot.withinss, mean(sil[, 3])))
}

# Best K by silhouette
optimal_k <- cluster_metrics$k[which.max(cluster_metrics$silhouette)]
best_sil  <- max(cluster_metrics$silhouette)

cat(strrep("-", 40), "\n")
cat(sprintf("🏆 Optimal K = %d  (Silhouette = %.4f)\n", optimal_k, best_sil))

In [ ]:
# ── Plot: Elbow + Silhouette ──────────────────────────────────────────────────

p_elbow <- ggplot(cluster_metrics, aes(x = k, y = wcss)) +
  geom_line(colour = "#1565C0", linewidth = 0.9) +
  geom_point(colour = "#1565C0", size = 3) +
  geom_vline(xintercept = optimal_k, colour = "#E53935",
             linetype = "dashed", linewidth = 0.8) +
  annotate("label", x = optimal_k, y = max(cluster_metrics$wcss, na.rm=TRUE) * 0.95,
           label = paste0("K = ", optimal_k), colour = "#E53935",
           fontface = "bold", size = 3.5) +
  scale_x_continuous(breaks = 2:K_MAX) +
  scale_y_continuous(labels = scales::comma) +
  labs(title    = "Elbow Method",
       subtitle = "Look for the 'elbow' — point of diminishing WCSS reduction",
       x = "Number of Clusters (K)", y = "Total Within-Cluster SS")

p_sil <- ggplot(cluster_metrics, aes(x = k, y = silhouette)) +
  geom_line(colour = "#2E7D32", linewidth = 0.9) +
  geom_point(colour = "#2E7D32", size = 3) +
  geom_vline(xintercept = optimal_k, colour = "#E53935",
             linetype = "dashed", linewidth = 0.8) +
  geom_point(data = cluster_metrics |> filter(k == optimal_k),
             colour = "#E53935", size = 5, shape = 21,
             fill = "#E53935", alpha = 0.5) +
  scale_x_continuous(breaks = 2:K_MAX) +
  labs(title    = "Silhouette Analysis",
       subtitle = "Higher = better cluster separation (max is optimal K)",
       x = "Number of Clusters (K)", y = "Average Silhouette Score")

gridExtra::grid.arrange(
  p_elbow, p_sil, nrow = 1,
  top = gridExtra::textGrob(
    paste0("K-Means Cluster Optimisation | Optimal K = ", optimal_k),
    gp = grid::gpar(fontface = "bold", fontsize = 13)
  )
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5.3 — Final K-Means Clustering on Full UMAP-10D
#
# nstart=25 runs 25 random initialisations and keeps the best solution,
# reducing sensitivity to centroid initialisation.
# ─────────────────────────────────────────────────────────────────────────────

set.seed(GLOBAL_SEED)
km_final <- kmeans(
  x        = umap_10d,
  centers  = optimal_k,
  nstart   = 25L,
  iter.max = 300L
)

# Add cluster labels to the main dataset
df$TextCluster <- as.integer(km_final$cluster)

cat("── Cluster Size Distribution ──\n")
cluster_sizes <- table(df$TextCluster)
print(cluster_sizes)
cat(sprintf("\nCluster balance: min=%d, max=%d (ratio=%.2f)\n",
            min(cluster_sizes), max(cluster_sizes),
            max(cluster_sizes) / min(cluster_sizes)))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5.4 — HDBSCAN Alternative (density-based clustering)
#
# HDBSCAN does not require specifying K in advance and handles irregular
# cluster shapes. Noise points are labelled -1.
# We include this as an alternative — compare its clusters to K-Means.
# ─────────────────────────────────────────────────────────────────────────────

py$umap_10d_np <- umap_10d

py_run_string("
import hdbscan
import numpy as np

print('Fitting HDBSCAN on UMAP-10D...')
clusterer = hdbscan.HDBSCAN(
    min_cluster_size     = 50,   # minimum viable cluster
    min_samples          = 5,    # core point density threshold
    metric               = 'euclidean',
    cluster_selection_method = 'eom',  # excess of mass
    core_dist_n_jobs     = -1    # parallelise
)
hdbscan_labels = clusterer.fit_predict(umap_10d_np)

n_clusters = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
n_noise    = int(np.sum(hdbscan_labels == -1))
pct_noise  = 100 * n_noise / len(hdbscan_labels)

print(f'HDBSCAN results:')
print(f'  Clusters found: {n_clusters}')
print(f'  Noise points:   {n_noise:,} ({pct_noise:.1f}%)')
print(f'  Probabilities:  mean={clusterer.probabilities_.mean():.3f}')
")

df$TextCluster_HDBSCAN <- as.integer(py$hdbscan_labels)

cat("\nHDBSCAN label distribution (-1 = noise):\n")
print(table(df$TextCluster_HDBSCAN))
cat("\n📌 K-Means clusters (TextCluster) used as primary feature for modelling.\n")
cat("   HDBSCAN labels (TextCluster_HDBSCAN) available for comparison.\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5.5 — Cluster Visualisations
# ─────────────────────────────────────────────────────────────────────────────

umap_vis <- tibble(
  UMAP1          = umap_2d[, 1],
  UMAP2          = umap_2d[, 2],
  KMeans         = factor(df$TextCluster),
  HDBSCAN        = factor(df$TextCluster_HDBSCAN),
  log_cost       = df$log_Target
)

p_km <- ggplot(umap_vis, aes(UMAP1, UMAP2, colour = KMeans)) +
  geom_point(alpha = 0.30, size = 0.5) +
  scale_colour_brewer(palette = "Set1") +
  guides(colour = guide_legend(override.aes = list(size = 3, alpha = 1))) +
  labs(title    = "K-Means Clusters",
       subtitle = paste0("K = ", optimal_k, " | BERT embeddings → UMAP 2D"),
       colour   = "Cluster")

p_hdb <- ggplot(umap_vis, aes(UMAP1, UMAP2, colour = HDBSCAN)) +
  geom_point(alpha = 0.30, size = 0.5) +
  scale_colour_viridis_d(option = "turbo") +
  guides(colour = guide_legend(override.aes = list(size = 3, alpha = 1))) +
  labs(title    = "HDBSCAN Clusters",
       subtitle = "Density-based | -1 = noise points",
       colour   = "Cluster")

p_cost <- ggplot(umap_vis, aes(UMAP1, UMAP2, colour = log_cost)) +
  geom_point(alpha = 0.30, size = 0.5) +
  scale_colour_viridis_c(option = "plasma", name = "log1p(Cost)") +
  labs(title    = "Claim Cost Gradient",
       subtitle = "High-cost claims cluster in specific semantic regions")

gridExtra::grid.arrange(
  p_km, p_hdb, p_cost, nrow = 1,
  top = gridExtra::textGrob(
    "UMAP 2D — BERT Embedding Space",
    gp = grid::gpar(fontface = "bold", fontsize = 13)
  )
)

In [ ]:
# ── Cluster profiling: cost and description examples ──────────────────────────
cluster_profile <- df |>
  group_by(TextCluster) |>
  summarise(
    N              = n(),
    MedianCost     = median(UltimateIncurredClaimCost),
    MeanCost       = mean(UltimateIncurredClaimCost),
    P90Cost        = quantile(UltimateIncurredClaimCost, 0.90),
    .groups        = "drop"
  ) |>
  arrange(desc(MedianCost))

cat("── Cluster Cost Profile ──\n")
cluster_profile |>
  mutate(across(c(MedianCost, MeanCost, P90Cost), scales::dollar)) |>
  print()

# Boxplot of cost by cluster
ggplot(df, aes(x = reorder(factor(TextCluster), UltimateIncurredClaimCost,
                            FUN = median),
               y = log_Target,
               fill = factor(TextCluster))) +
  geom_boxplot(alpha = 0.75, outlier.size = 0.4, outlier.alpha = 0.3) +
  scale_fill_brewer(palette = "Set1", guide = "none") +
  labs(
    title    = "Claim Cost Distribution by Text Cluster",
    subtitle = "Clusters ordered by median cost — confirms semantic groupings differ in cost",
    x        = "Text Cluster (ordered by median cost)",
    y        = "log1p(UltimateIncurredClaimCost)"
  )

In [ ]:
# ── Top description examples per cluster ─────────────────────────────────────
cat("── Sample Claim Descriptions by Cluster ──\n\n")
for (k in sort(unique(df$TextCluster))) {
  examples <- df |>
    filter(TextCluster == k, nchar(ClaimDescription) > 10) |>
    slice_sample(n = 4) |>
    pull(ClaimDescription)
  med_cost <- cluster_profile |> filter(TextCluster == k) |> pull(MedianCost)
  cat(sprintf("Cluster %d  [median cost: %s]\n",
              k, scales::dollar(round(med_cost, 0))))
  purrr::walk(examples, ~ cat(sprintf("  → %s\n", .x)))
  cat("\n")
}

---
## Section 6 — Train / Test Split & Feature Matrix Construction

We create a clean, leakage-free modeling dataset:
1. Define the exact feature set (no identifiers, no raw dates, no target-derived columns)
2. Perform a **stratified 80/20 split** — stratified on `TextCluster` to ensure cluster representation in both sets
3. Recompute imputation statistics from training data only
4. Convert to `xgb.DMatrix` objects for efficient XGBoost training

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6.1 — Define Feature Set
#
# Explicitly list all model features. No identifiers, no raw text, no raw dates,
# no target variable, no HDBSCAN label (excluded to keep model clean).
#
# TextCluster is the NLP-derived feature that bridges text → model.
# ─────────────────────────────────────────────────────────────────────────────

FEATURES <- c(
  # ── Claimant demographics ─────────────────────────────────────────────────
  "Age",                 # claimant age at accident
  "Gender_enc",          # encoded: M/F/Unknown
  "MaritalStatus_enc",   # encoded: M/S/U/D/Unknown
  "DependentChildren",   # integer count
  "DependentsOther",     # integer count

  # ── Employment characteristics ────────────────────────────────────────────
  "log_WeeklyWages",     # log-transformed pre-injury wages
  "PartTimeFullTime_enc",# encoded employment type
  "HoursWorkedPerWeek",  # weekly hours
  "DaysWorkedPerWeek",   # days per week

  # ── Claim financials (early estimate — leakage discussed in §2) ───────────
  "log_InitialCost",     # log(InitialIncurredCalimsCost)

  # ── Temporal features ─────────────────────────────────────────────────────
  "ReportingDelay",      # days between accident and report
  "AccidentYear",        # development year
  "AccidentMonth",       # seasonal patterns
  "AccidentWeekday",     # day of week (1=Sun … 7=Sat)
  "AccidentHour",        # hour of accident
  "IsWeekend",           # binary weekend flag

  # ── NLP feature (BERT → UMAP → K-Means) ──────────────────────────────────
  "TextCluster"          # semantic injury type cluster label
)

TARGET <- "log_Target"  # log1p(UltimateIncurredClaimCost) — back-transformed for evaluation

# Assemble modeling dataframe
df_model <- df |>
  select(all_of(c(FEATURES, TARGET))) |>
  drop_na()

cat(sprintf("✅ Model feature matrix: %s rows × %d features\n",
            scales::comma(nrow(df_model)), length(FEATURES)))
cat("\nFeatures included:\n")
purrr::walk(FEATURES, ~ cat(sprintf("  ✓ %s\n", .x)))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6.2 — Stratified Train/Test Split
#
# caret::createDataPartition() performs stratified sampling to maintain
# the proportional distribution of TextCluster across train and test.
#
# LEAKAGE SAFEGUARD: All subsequent preprocessing (if any) MUST be fitted
# on train_df only, then applied to test_df.
# ─────────────────────────────────────────────────────────────────────────────

set.seed(GLOBAL_SEED)
train_idx <- caret::createDataPartition(
  y    = df_model$TextCluster,
  p    = CONFIG$train_frac,
  list = FALSE
)

train_df <- df_model[ train_idx, ]
test_df  <- df_model[-train_idx, ]

cat("── Train / Test Split ──\n")
cat(sprintf("  Train: %s rows (%.0f%%)\n",
            scales::comma(nrow(train_df)), CONFIG$train_frac * 100))
cat(sprintf("  Test:  %s rows (%.0f%%)\n",
            scales::comma(nrow(test_df)), (1 - CONFIG$train_frac) * 100))

# Verify stratification
cat("\n── Cluster distribution check (should match train≈test) ──\n")
prop_train <- prop.table(table(train_df$TextCluster))
prop_test  <- prop.table(table(test_df$TextCluster))
stratification_check <- tibble(
  cluster    = names(prop_train),
  train_pct  = round(as.numeric(prop_train) * 100, 1),
  test_pct   = round(as.numeric(prop_test)  * 100, 1)
)
print(stratification_check)

# ── Convert to XGBoost DMatrix ────────────────────────────────────────────────
X_train <- as.matrix(train_df[, FEATURES])
y_train <- train_df[[TARGET]]
X_test  <- as.matrix(test_df[, FEATURES])
y_test  <- test_df[[TARGET]]

dtrain  <- xgboost::xgb.DMatrix(data = X_train, label = y_train)
dtest   <- xgboost::xgb.DMatrix(data = X_test,  label = y_test)

cat("\n✅ xgb.DMatrix objects created.\n")

---
## Section 7 — XGBoost Model: Hyperparameter Tuning & Cross-Validation

We use **XGBoost** (eXtreme Gradient Boosting) — the standard for tabular actuarial modelling.

### Why XGBoost?
- Handles mixed data types (numeric, categorical integer codes) natively
- Robust to outliers via tree structure
- Built-in regularisation (L1 alpha, L2 lambda) prevents overfitting
- Natively supports early stopping with cross-validation
- SHAP values available via `SHAPforxgboost` without additional approximations

### Hyperparameter Search Strategy
We perform a **grid search with 5-fold cross-validation** and early stopping:

| Parameter | Values Searched | Description |
|---|---|---|
| `max_depth` | 4, 6, 8 | Tree depth — higher = more complex |
| `eta` | 0.05, 0.10 | Learning rate — lower = better generalisation |
| `subsample` | 0.70, 0.90 | Row subsampling fraction |
| `colsample_bytree` | 0.70, 0.90 | Column subsampling per tree |
| `min_child_weight` | 5, 10 | Minimum sum of instance weights per leaf |

> **Note:** `nrounds` is determined automatically via early stopping (patience = 30 rounds without improvement on the CV validation fold), preventing overfitting on the training set.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7.1 — Hyperparameter Grid Search
#
# expand.grid() creates all combinations. For each, xgb.cv() runs
# k-fold cross-validation and returns per-round test metrics.
# We extract the best round and its CV RMSE for comparison.
# ─────────────────────────────────────────────────────────────────────────────

param_grid <- expand.grid(
  max_depth         = c(4L, 6L, 8L),
  eta               = c(0.05, 0.10),
  subsample         = c(0.70, 0.90),
  colsample_bytree  = c(0.70, 0.90),
  min_child_weight  = c(5L, 10L),
  stringsAsFactors  = FALSE
)

cat(sprintf("Grid size: %d parameter combinations\n", nrow(param_grid)))
cat(sprintf("CV folds:  %d | Early stopping: %d rounds\n\n",
            CONFIG$cv_folds, CONFIG$early_stop))

# Initialise results storage
cv_results <- vector("list", nrow(param_grid))

set.seed(GLOBAL_SEED)
for (i in seq_len(nrow(param_grid))) {
  params_i <- list(
    booster          = "gbtree",
    objective        = "reg:squarederror",
    max_depth        = param_grid$max_depth[i],
    eta              = param_grid$eta[i],
    subsample        = param_grid$subsample[i],
    colsample_bytree = param_grid$colsample_bytree[i],
    min_child_weight = param_grid$min_child_weight[i],
    lambda           = 1.0,     # L2 regularisation
    alpha            = 0.1,     # L1 regularisation
    tree_method      = "hist",  # faster histogram method
    eval_metric      = "rmse"
  )

  cv_i <- xgboost::xgb.cv(
    params                = params_i,
    data                  = dtrain,
    nrounds               = CONFIG$max_nrounds,
    nfold                 = CONFIG$cv_folds,
    verbose               = 0,
    early_stopping_rounds = CONFIG$early_stop,
    maximize              = FALSE,
    seed                  = GLOBAL_SEED
  )

  best_round <- which.min(cv_i$evaluation_log$test_rmse_mean)
  best_rmse  <- cv_i$evaluation_log$test_rmse_mean[best_round]
  best_sd    <- cv_i$evaluation_log$test_rmse_std[best_round]

  cv_results[[i]] <- c(
    as.list(param_grid[i, ]),
    list(best_rmse = best_rmse, best_sd = best_sd, best_nround = best_round)
  )

  cat(sprintf("[%3d/%d] d=%d η=%.2f sub=%.1f col=%.1f mcw=%2d | "
              "CV-RMSE=%.5f±%.5f  rounds=%d\n",
              i, nrow(param_grid),
              param_grid$max_depth[i], param_grid$eta[i],
              param_grid$subsample[i], param_grid$colsample_bytree[i],
              param_grid$min_child_weight[i],
              best_rmse, best_sd, best_round))
}

cat("\n✅ Grid search complete.\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7.2 — Select Best Configuration & Train Final Model
# ─────────────────────────────────────────────────────────────────────────────

cv_df      <- bind_rows(lapply(cv_results, as_tibble))
best_row   <- cv_df |> slice_min(best_rmse, n = 1)

cat("── Top 5 Configurations by CV-RMSE ──\n")
cv_df |>
  arrange(best_rmse) |>
  slice_head(n = 5) |>
  print()

cat("\n🏆 Best configuration:\n")
print(best_row)

# ── Build best params list ────────────────────────────────────────────────────
best_params <- list(
  booster          = "gbtree",
  objective        = "reg:squarederror",
  max_depth        = best_row$max_depth,
  eta              = best_row$eta,
  subsample        = best_row$subsample,
  colsample_bytree = best_row$colsample_bytree,
  min_child_weight = best_row$min_child_weight,
  lambda           = 1.0,
  alpha            = 0.1,
  tree_method      = "hist",
  eval_metric      = "rmse"
)
best_nrounds <- best_row$best_nround

# ── Train final model on full training set ────────────────────────────────────
cat(sprintf("\nTraining final model: %d rounds, depth=%d, η=%.2f\n",
            best_nrounds, best_params$max_depth, best_params$eta))

set.seed(GLOBAL_SEED)
xgb_final <- xgboost::xgb.train(
  params        = best_params,
  data          = dtrain,
  nrounds       = best_nrounds,
  watchlist     = list(train = dtrain, eval = dtest),
  verbose       = 1,
  print_every_n = max(1L, best_nrounds %/% 10L)  # print ~10 updates
)

cat("\n✅ Final XGBoost model trained.\n")

In [ ]:
# ── CV RMSE improvement curve for best model ──────────────────────────────────

# Re-run CV for the best params to extract the learning curve
set.seed(GLOBAL_SEED)
cv_best <- xgboost::xgb.cv(
  params                = best_params,
  data                  = dtrain,
  nrounds               = CONFIG$max_nrounds,
  nfold                 = CONFIG$cv_folds,
  verbose               = 0,
  early_stopping_rounds = CONFIG$early_stop,
  maximize              = FALSE,
  seed                  = GLOBAL_SEED
)

cv_log <- cv_best$evaluation_log

ggplot(cv_log, aes(x = iter)) +
  geom_ribbon(aes(ymin = train_rmse_mean - train_rmse_std,
                  ymax = train_rmse_mean + train_rmse_std),
              fill = "#1565C0", alpha = 0.20) +
  geom_line(aes(y = train_rmse_mean, colour = "Train"), linewidth = 0.8) +
  geom_ribbon(aes(ymin = test_rmse_mean - test_rmse_std,
                  ymax = test_rmse_mean + test_rmse_std),
              fill = "#E53935", alpha = 0.20) +
  geom_line(aes(y = test_rmse_mean, colour = "CV Test"), linewidth = 0.8) +
  geom_vline(xintercept = best_nrounds, linetype = "dashed",
             colour = "grey40", linewidth = 0.7) +
  annotate("label", x = best_nrounds, y = max(cv_log$test_rmse_mean) * 0.98,
           label = paste0("Best round: ", best_nrounds),
           size = 3.5, colour = "grey40") +
  scale_colour_manual(values = c("Train" = "#1565C0", "CV Test" = "#E53935")) +
  labs(
    title    = "XGBoost Learning Curve (Best Configuration)",
    subtitle = paste0("5-fold CV RMSE ± 1SD | Best params: depth=",
                      best_params$max_depth, ", η=", best_params$eta),
    x        = "Boosting Round",
    y        = "RMSE (log scale)",
    colour   = NULL
  )

---
## Section 8 — Model Evaluation

We evaluate the final model on the **held-out test set** (never seen during training or hyperparameter tuning).

### Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **RMSE** | √(mean((y−ŷ)²)) | Average error magnitude; penalises large errors |
| **MAE** | mean(|y−ŷ|) | Average absolute error; robust to outliers |
| **R²** | 1 − SS_res/SS_tot | Proportion of variance explained (0–1) |

All metrics are reported on both the **original dollar scale** (for actuarial interpretation) and the **log scale** (for model diagnostics).

**Back-transformation:** Since we modelled `log1p(cost)`, predictions are converted back via `expm1(ŷ)` = exp(ŷ) − 1.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8.1 — Generate Predictions & Compute Metrics
# ─────────────────────────────────────────────────────────────────────────────

# Predict on log scale, back-transform to dollars
pred_log    <- predict(xgb_final, dtest)
pred_dollar <- expm1(pred_log)                   # inverse of log1p
actual_log  <- y_test
actual_dollar <- expm1(actual_log)

# ── Helper: compute all three metrics ────────────────────────────────────────
compute_metrics <- function(actual, predicted, label = "") {
  rmse  <- sqrt(mean((actual - predicted)^2))
  mae   <- mean(abs(actual - predicted))
  ss_r  <- sum((actual - predicted)^2)
  ss_t  <- sum((actual - mean(actual))^2)
  r2    <- 1 - ss_r / ss_t
  tibble(Scale = label, RMSE = rmse, MAE = mae, R2 = r2)
}

metrics_log    <- compute_metrics(actual_log,    pred_log,    "Log Scale")
metrics_dollar <- compute_metrics(actual_dollar, pred_dollar, "Dollar Scale")

cat("══════════════════════════════════════════════════════\n")
cat("  FINAL MODEL EVALUATION — HOLD-OUT TEST SET\n")
cat("══════════════════════════════════════════════════════\n")
cat("  Test set size:", scales::comma(nrow(test_df)), "claims\n\n")
cat("  ── Original Dollar Scale ──\n")
cat(sprintf("     RMSE: %s\n",  scales::dollar(round(metrics_dollar$RMSE, 0))))
cat(sprintf("     MAE:  %s\n",  scales::dollar(round(metrics_dollar$MAE,  0))))
cat(sprintf("     R²:    %.4f\n", metrics_dollar$R2))
cat("\n  ── Log Scale (model native) ──\n")
cat(sprintf("     RMSE: %.5f\n", metrics_log$RMSE))
cat(sprintf("     MAE:  %.5f\n", metrics_log$MAE))
cat(sprintf("     R²:    %.4f\n", metrics_log$R2))
cat("══════════════════════════════════════════════════════\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8.2 — Diagnostic Plots
# ─────────────────────────────────────────────────────────────────────────────

eval_df <- tibble(
  actual_log    = actual_log,
  pred_log      = pred_log,
  actual_dollar = actual_dollar,
  pred_dollar   = pred_dollar,
  residual_log  = actual_log    - pred_log,
  residual_pct  = (actual_dollar - pred_dollar) / actual_dollar * 100,
  cluster       = factor(test_df$TextCluster)
)

# 1. Predicted vs Actual (log scale)
p1 <- ggplot(eval_df, aes(x = pred_log, y = actual_log, colour = cluster)) +
  geom_point(alpha = 0.25, size = 0.7) +
  geom_abline(slope = 1, intercept = 0, colour = "black",
              linewidth = 1, linetype = "dashed") +
  scale_colour_brewer(palette = "Set1") +
  guides(colour = guide_legend(override.aes = list(size = 3, alpha = 1))) +
  labs(title    = "Predicted vs Actual (Log Scale)",
       subtitle = sprintf("R² = %.4f  |  RMSE = %.4f",
                          metrics_log$R2, metrics_log$RMSE),
       x = "log1p(Predicted)", y = "log1p(Actual)", colour = "Cluster")

# 2. Residuals vs Fitted (log scale)
p2 <- ggplot(eval_df, aes(x = pred_log, y = residual_log, colour = cluster)) +
  geom_point(alpha = 0.25, size = 0.7) +
  geom_hline(yintercept = 0, colour = "black", linewidth = 0.9) +
  geom_smooth(method = "loess", se = FALSE, colour = "#E53935",
              linewidth = 0.8, show.legend = FALSE) +
  scale_colour_brewer(palette = "Set1") +
  labs(title    = "Residuals vs Fitted",
       subtitle = "Red curve: LOESS smoother (should be flat at 0)",
       x = "log1p(Predicted)", y = "Residual", colour = "Cluster")

# 3. Residual distribution
p3 <- ggplot(eval_df, aes(x = residual_log)) +
  geom_histogram(bins = 80, fill = "#5C6BC0", alpha = 0.85, colour = "white") +
  geom_vline(xintercept = 0, colour = "#E53935", linewidth = 0.9) +
  geom_vline(xintercept = mean(eval_df$residual_log),
             colour = "#2E7D32", linetype = "dashed", linewidth = 0.8) +
  labs(title    = "Residual Distribution",
       subtitle = sprintf("Mean residual = %.5f (should be ≈ 0)",
                          mean(eval_df$residual_log)),
       x = "Residual (log scale)", y = "Count")

# 4. Percentage error by cluster
p4 <- ggplot(eval_df,
             aes(x = reorder(cluster, abs(residual_pct), FUN = median),
                 y = abs(residual_pct),
                 fill = cluster)) +
  geom_boxplot(alpha = 0.7, outlier.size = 0.4, outlier.alpha = 0.3) +
  scale_fill_brewer(palette = "Set1", guide = "none") +
  scale_y_continuous(limits = c(0, 200),
                     labels = scales::percent_format(scale = 1)) +
  labs(title    = "Absolute % Error by Cluster",
       subtitle = "Clusters with larger errors may benefit from cluster-specific models",
       x = "Text Cluster", y = "| % Error |")

gridExtra::grid.arrange(
  p1, p2, p3, p4, nrow = 2,
  top = gridExtra::textGrob(
    "XGBoost Model Diagnostics — Hold-Out Test Set",
    gp = grid::gpar(fontface = "bold", fontsize = 13)
  )
)

In [ ]:
# ── Per-cluster performance table ─────────────────────────────────────────────
cat("── Model Performance by TextCluster ──\n")

eval_df |>
  group_by(cluster) |>
  summarise(
    N         = n(),
    RMSE_log  = round(sqrt(mean(residual_log^2)), 4),
    MAE_log   = round(mean(abs(residual_log)), 4),
    R2_log    = round(1 - sum(residual_log^2) /
                      sum((actual_log - mean(actual_log))^2), 4),
    MedActual = scales::dollar(round(median(actual_dollar), 0)),
    .groups = "drop"
  ) |>
  arrange(desc(RMSE_log)) |>
  print()

---
## Section 9 — Explainable AI with SHAP Values

**SHAP (SHapley Additive exPlanations)** is the gold standard for interpreting tree-based models. SHAP values are grounded in cooperative game theory and have the following properties:

- **Consistency:** If a feature's contribution increases, its SHAP value increases
- **Local accuracy:** SHAP values sum exactly to the model's prediction
- **Missingness:** Features not contributing receive SHAP = 0

### SHAP Analyses Performed

| Analysis | What it shows |
|---|---|
| **Global importance bar** | Mean \|SHAP\| per feature — overall driver ranking |
| **Beeswarm summary** | Distribution of SHAP values — direction and magnitude |
| **Dependence plots** | How feature value affects its SHAP value |
| **Waterfall (per-claim)** | Which features pushed a specific claim cost up/down |
| **Force plot** | Compact version of waterfall |

> **Interpretation tip:** Positive SHAP = the feature increased predicted cost above the baseline (mean prediction). Negative SHAP = the feature decreased it.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9.1 — Compute SHAP Values
#
# SHAPforxgboost::shap.values() uses XGBoost's internal TreeSHAP algorithm
# (exact, not approximate) for O(TLD) computation where T=trees, L=leaves, D=depth.
#
# We use a 5,000-observation sample to keep runtime reasonable.
# For production, compute on the full training set.
# ─────────────────────────────────────────────────────────────────────────────

library(SHAPforxgboost)
library(shapviz)

# Draw sample for SHAP computation
set.seed(GLOBAL_SEED)
shap_n    <- min(CONFIG$shap_n_sample, nrow(X_train))
shap_idx  <- sample(nrow(X_train), shap_n)
X_shap    <- X_train[shap_idx, ]

cat(sprintf("Computing SHAP values for %s observations...\n",
            scales::comma(shap_n)))

# Compute SHAP values using TreeSHAP
shap_out  <- SHAPforxgboost::shap.values(
  xgb_model = xgb_final,
  X_train   = X_shap
)

# Prepare long-format for ggplot
shap_long <- SHAPforxgboost::shap.prep(
  xgb_model = xgb_final,
  X_train   = X_shap
)

# Build shapviz object (for waterfall / force / dependence plots)
sv_train <- shapviz::shapviz(
  object = xgb_final,
  X_pred = xgboost::xgb.DMatrix(X_shap),
  X      = X_shap
)

# Build shapviz for test set (for individual claim explanations)
sv_test  <- shapviz::shapviz(
  object = xgb_final,
  X_pred = dtest,
  X      = X_test
)

cat("✅ SHAP values computed.\n")
cat(sprintf("   SHAP matrix: %d obs × %d features\n",
            nrow(shap_out$shap_score), ncol(shap_out$shap_score)))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9.2 — Global Feature Importance (SHAP)
#
# Mean |SHAP| ranks features by their average absolute contribution.
# This is more reliable than XGBoost's native gain importance because it
# measures the actual effect on the model output, not the training loss.
# ─────────────────────────────────────────────────────────────────────────────

# Bar chart: mean |SHAP|
shap_importance <- shap_out$mean_shap_score |>
  as_tibble(rownames = "feature") |>
  rename(mean_abs_shap = value) |>
  arrange(desc(mean_abs_shap))

cat("── Global SHAP Importance (mean |SHAP|) ──\n")
print(shap_importance)

ggplot(shap_importance, aes(x = reorder(feature, mean_abs_shap),
                             y = mean_abs_shap,
                             fill = mean_abs_shap)) +
  geom_col(width = 0.7) +
  coord_flip() +
  scale_fill_viridis_c(option = "viridis", guide = "none") +
  geom_text(aes(label = round(mean_abs_shap, 4)),
            hjust = -0.1, size = 3.2, colour = "grey30") +
  labs(
    title    = "Global SHAP Feature Importance",
    subtitle = "Mean absolute SHAP value across training sample",
    x        = NULL,
    y        = "Mean |SHAP| (log scale contribution)",
    caption  = paste("Based on", scales::comma(shap_n), "training observations")
  )

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9.3 — SHAP Summary Plot (Beeswarm)
#
# The beeswarm plot shows BOTH magnitude AND direction of each feature's impact:
#   • X-axis: SHAP value (positive = increases predicted cost)
#   • Colour: actual feature value (red = high, blue = low)
#
# This is the most information-dense SHAP visualisation.
# ─────────────────────────────────────────────────────────────────────────────

# SHAPforxgboost native beeswarm
SHAPforxgboost::shap.plot.summary(
  shap_long,
  top_n      = length(FEATURES),
  scientific = FALSE
) +
  labs(
    title    = "SHAP Summary — Beeswarm Plot",
    subtitle = paste0("Each point = one claim | ",
                      "Colour = feature value (red=high, blue=low)"),
    caption  = "Positive SHAP → increases predicted log(cost) | Negative → decreases"
  ) +
  theme_actuarial()

In [ ]:
# ── shapviz beeswarm (alternative styling) ────────────────────────────────────
shapviz::sv_importance(sv_train, kind = "beeswarm", max_display = 15) +
  labs(
    title    = "SHAP Beeswarm — shapviz",
    subtitle = "Top 15 features by mean |SHAP|"
  ) +
  theme_actuarial()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9.4 — SHAP Dependence Plots (Top 4 Features)
#
# Dependence plots show how the SHAP value for one feature changes as its
# value changes, while interaction with another feature is shown via colour.
# These reveal non-linear relationships and interactions.
# ─────────────────────────────────────────────────────────────────────────────

top4_features <- shap_importance$feature[1:4]
cat("Creating dependence plots for:", paste(top4_features, collapse = ", "), "\n")

dep_plots <- purrr::map(top4_features, function(feat) {
  # auto colour_var selects the feature with highest interaction
  shapviz::sv_dependence(sv_train, v = feat, color_var = "auto") +
    theme_actuarial(base_size = 10)
})

gridExtra::grid.arrange(
  grobs = dep_plots, nrow = 2,
  top = gridExtra::textGrob(
    "SHAP Dependence Plots — Top 4 Features",
    gp = grid::gpar(fontface = "bold", fontsize = 13)
  )
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9.5 — SHAP Dependence: TextCluster × InitialCost Interaction
#
# This specific plot examines whether the NLP-derived TextCluster modifies
# the relationship between initial cost estimate and final claim cost.
# Parallel lines = no interaction; diverging lines = interaction present.
# ─────────────────────────────────────────────────────────────────────────────

SHAPforxgboost::shap.plot.dependence(
  data_long     = shap_long,
  x             = "log_InitialCost",
  y             = "log_InitialCost",
  color_feature = "TextCluster"
) +
  scale_colour_viridis_c(option = "turbo") +
  labs(
    title    = "SHAP Dependence: log_InitialCost × TextCluster",
    subtitle = "Colour = TextCluster value — reveals cluster-specific cost dynamics",
    x        = "log_InitialCost (feature value)",
    y        = "SHAP value for log_InitialCost"
  ) +
  theme_actuarial()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9.6 — Individual Claim Explanations (Waterfall Plots)
#
# We explain 3 representative claims from the test set:
#   • Low-cost claim  (10th percentile)
#   • Median claim    (50th percentile)
#   • High-cost claim (90th percentile)
#
# The waterfall plot shows: baseline (mean prediction) → feature contributions
# → final prediction. Red = pushes cost up, Blue = pushes cost down.
# ─────────────────────────────────────────────────────────────────────────────

# Find representative test-set indices
q10_idx  <- which.min(abs(actual_dollar - quantile(actual_dollar, 0.10)))
q50_idx  <- which.min(abs(actual_dollar - quantile(actual_dollar, 0.50)))
q90_idx  <- which.min(abs(actual_dollar - quantile(actual_dollar, 0.90)))

claim_indices <- c(q10_idx, q50_idx, q90_idx)
claim_labels  <- c("Low-Cost Claim (P10)",
                   "Median Claim (P50)",
                   "High-Cost Claim (P90)")

for (j in seq_along(claim_indices)) {
  idx       <- claim_indices[j]
  act_str   <- scales::dollar(round(actual_dollar[idx], 0))
  pred_str  <- scales::dollar(round(pred_dollar[idx],  0))
  err_str   <- scales::dollar(round(abs(actual_dollar[idx] - pred_dollar[idx]), 0))

  p <- shapviz::sv_waterfall(sv_test, row_id = idx, max_display = 12) +
    labs(
      title    = paste0("SHAP Waterfall — ", claim_labels[j]),
      subtitle = sprintf("Actual: %s  |  Predicted: %s  |  Error: %s",
                         act_str, pred_str, err_str),
      caption  = paste0("ClaimDescription: ",
                         substr(df$ClaimDescription[
                           which(!is.na(df$log_Target))[nrow(train_df) + idx]], 1, 80))
    ) +
    theme_actuarial(base_size = 11)

  print(p)
  cat("\n")
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9.7 — SHAP Force Plot (High-Cost Claim)
#
# The force plot is a compact alternative to the waterfall, showing all
# features that push the prediction above (red) or below (blue) the baseline.
# ─────────────────────────────────────────────────────────────────────────────

shapviz::sv_force(sv_test, row_id = q90_idx) +
  labs(
    title    = "SHAP Force Plot — High-Cost Claim",
    subtitle = sprintf("Actual: %s  |  Predicted: %s",
                       scales::dollar(round(actual_dollar[q90_idx], 0)),
                       scales::dollar(round(pred_dollar[q90_idx], 0)))
  ) +
  theme_actuarial(base_size = 11)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9.8 — SHAP: TextCluster Feature Contribution Analysis
#
# This analysis specifically examines the actuarial value added by the
# NLP-derived TextCluster feature — does text clustering meaningfully
# improve predictions beyond structured features alone?
# ─────────────────────────────────────────────────────────────────────────────

# Extract SHAP values for TextCluster feature across all SHAP sample
shap_cluster_vals <- shap_out$shap_score[, "TextCluster"]
cluster_feature_vals <- X_shap[, "TextCluster"]

# SHAP value of TextCluster by cluster group
cluster_shap_df <- tibble(
  cluster   = factor(cluster_feature_vals),
  shap_val  = shap_cluster_vals
)

ggplot(cluster_shap_df, aes(x = cluster, y = shap_val, fill = cluster)) +
  geom_boxplot(alpha = 0.7, outlier.size = 0.4) +
  geom_hline(yintercept = 0, colour = "black", linewidth = 0.8, linetype = "dashed") +
  scale_fill_brewer(palette = "Set1", guide = "none") +
  labs(
    title    = "SHAP Value of TextCluster Feature by Cluster Group",
    subtitle = paste0("Positive SHAP = cluster increases predicted cost ",
                      "| Negative = decreases cost"),
    x        = "TextCluster Label",
    y        = "SHAP value (log scale contribution)",
    caption  = "Validates that NLP clustering captures cost-relevant injury semantics"
  )

---
## Section 10 — Model Persistence & Final Summary

We save all outputs for reproducibility and downstream use.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10.1 — Save Model & Artefacts
# ─────────────────────────────────────────────────────────────────────────────

# Save XGBoost model (JSON format — portable and version-stable)
xgboost::xgb.save(xgb_final, CONFIG$model_path)
cat(sprintf("✅ Model saved: %s\n", CONFIG$model_path))

# Save predictions with metadata
predictions_out <- test_df |>
  select(TextCluster) |>
  mutate(
    actual_log_cost      = y_test,
    predicted_log_cost   = pred_log,
    actual_dollar_cost   = expm1(y_test),
    predicted_dollar_cost= pred_dollar,
    residual_log         = y_test - pred_log,
    pct_error            = (expm1(y_test) - pred_dollar) / expm1(y_test) * 100
  )
readr::write_csv(predictions_out, CONFIG$pred_path)
cat(sprintf("✅ Predictions saved: %s\n", CONFIG$pred_path))

# Save encoding maps
saveRDS(encoding_maps, "encoding_maps.rds")
cat("✅ Encoding maps saved: encoding_maps.rds\n")

# Save cluster centroids (for scoring new data)
saveRDS(list(
  km_model    = km_final,
  optimal_k   = optimal_k,
  bert_model  = CONFIG$bert_model
), "clustering_model.rds")
cat("✅ Clustering model saved: clustering_model.rds\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10.2 — Final Summary Dashboard
# ─────────────────────────────────────────────────────────────────────────────

cat("\n")
cat("╔══════════════════════════════════════════════════════════════════════╗\n")
cat("║           ACTUARIAL ULTIMATE LOSS PREDICTION — FINAL SUMMARY         ║\n")
cat("╠══════════════════════════════════════════════════════════════════════╣\n")
cat("║  DATASET                                                              ║\n")
cat(sprintf("║    Total claims:         %-10s                               ║\n",
            scales::comma(nrow(df_raw))))
cat(sprintf("║    Training set:         %-10s                               ║\n",
            scales::comma(nrow(train_df))))
cat(sprintf("║    Test set:             %-10s                               ║\n",
            scales::comma(nrow(test_df))))
cat("╠══════════════════════════════════════════════════════════════════════╣\n")
cat("║  NLP PIPELINE                                                         ║\n")
cat(sprintf("║    BERT model:           %-36s ║\n", CONFIG$bert_model))
cat(sprintf("║    Embedding dims:       384 → UMAP-10D → K-Means            ║\n"))
cat(sprintf("║    Text clusters (K):    %-10d                               ║\n", optimal_k))
cat(sprintf("║    Best silhouette:      %-10.4f                               ║\n", best_sil))
cat("╠══════════════════════════════════════════════════════════════════════╣\n")
cat("║  MODEL                                                                ║\n")
cat("║    Algorithm:            XGBoost (reg:squarederror)                   ║\n")
cat("║    Target:               log1p(UltimateIncurredClaimCost)             ║\n")
cat(sprintf("║    Features:             %-10d                               ║\n", length(FEATURES)))
cat(sprintf("║    Boosting rounds:      %-10d                               ║\n", best_nrounds))
cat(sprintf("║    max_depth:            %-10d                               ║\n", best_params$max_depth))
cat(sprintf("║    learning rate (η):    %-10.2f                               ║\n", best_params$eta))
cat(sprintf("║    subsample:            %-10.2f                               ║\n", best_params$subsample))
cat(sprintf("║    colsample_bytree:     %-10.2f                               ║\n", best_params$colsample_bytree))
cat("╠══════════════════════════════════════════════════════════════════════╣\n")
cat("║  PERFORMANCE (HOLD-OUT TEST SET)                                      ║\n")
cat(sprintf("║    RMSE (dollars):       %-30s      ║\n",
            scales::dollar(round(metrics_dollar$RMSE, 0))))
cat(sprintf("║    MAE  (dollars):       %-30s      ║\n",
            scales::dollar(round(metrics_dollar$MAE, 0))))
cat(sprintf("║    R² (dollar scale):    %-10.4f                               ║\n",
            metrics_dollar$R2))
cat(sprintf("║    RMSE (log scale):     %-10.5f                               ║\n",
            metrics_log$RMSE))
cat(sprintf("║    R² (log scale):       %-10.4f                               ║\n",
            metrics_log$R2))
cat("╠══════════════════════════════════════════════════════════════════════╣\n")
cat("║  TOP 3 SHAP FEATURES                                                  ║\n")
for (i in 1:3) {
  cat(sprintf("║    %d. %-20s (mean|SHAP|=%.5f)                     ║\n",
              i, shap_importance$feature[i],
              shap_importance$mean_abs_shap[i]))
}
cat("╠══════════════════════════════════════════════════════════════════════╣\n")
cat("║  OUTPUTS                                                              ║\n")
cat(sprintf("║    %-66s║\n", paste0("Model:          ", CONFIG$model_path)))
cat(sprintf("║    %-66s║\n", paste0("Predictions:    ", CONFIG$pred_path)))
cat(sprintf("║    %-66s║\n", "Clustering:     clustering_model.rds"))
cat(sprintf("║    %-66s║\n", "Encodings:      encoding_maps.rds"))
cat("╚══════════════════════════════════════════════════════════════════════╝\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10.3 — Session Info (Reproducibility Record)
#
# Record exact package versions and R version used. Essential for
# reproducible research — include this in any published analysis.
# ─────────────────────────────────────────────────────────────────────────────

cat("\n── R Session Information ──\n")
sessionInfo()

---
## Appendix A — Scoring New Claims

To score a new batch of claims using the saved model:

```r
# 1. Load saved artefacts
xgb_model      <- xgboost::xgb.load("xgb_model.json")
encoding_maps  <- readRDS("encoding_maps.rds")
cluster_model  <- readRDS("clustering_model.rds")

# 2. Apply the same preprocessing pipeline to new data
#    (impute medians from TRAINING set, not new data)

# 3. Generate BERT embeddings → UMAP → K-Means predict()
#    Note: use predict() on the saved km_model, NOT re-fit
new_cluster <- predict(cluster_model$km_model, new_umap_10d)$cluster

# 4. Score
new_X    <- as.matrix(new_df[, FEATURES])
new_dmat <- xgboost::xgb.DMatrix(new_X)
pred_log <- predict(xgb_model, new_dmat)
pred_cost <- expm1(pred_log)  # back to dollar scale
```

> **Important:** Always apply the *training set* imputation medians and the *fitted* UMAP/K-Means models to new data. Never re-fit these on new data.

---

## Appendix B — Actuarial Considerations

### Model Limitations
1. **Temporal validation:** This notebook uses random train/test split. For actuarial use, consider **out-of-time validation** (train on older accident years, test on most recent).
2. **Tail risk:** RMSE penalises large errors, but actuaries often care more about the 95th+ percentile of predictions. Consider quantile regression or a separate severity model for large claims.
3. **InitialIncurredCalimsCost:** This feature dominates predictions due to its correlation with the target. If using the model for **prospective pricing** (before initial estimate is set), remove this feature and retrain.
4. **Regulatory explainability:** SHAP values provide claim-level explanations suitable for regulatory scrutiny and adverse action notices.

### Potential Extensions
- **Two-part model:** Separate frequency (logistic) and severity (this model) components
- **Tweedie regression:** Native support for insurance claim distributions in XGBoost via `objective='reg:tweedie'`
- **Ensemble:** Stack XGBoost predictions with a GLM for regulatory interpretability
- **BERT fine-tuning:** Fine-tune the sentence encoder on insurance text for domain adaptation